# 01 — Diagnostic du modèle Gradient Boosting de référence

Ce notebook charge uniquement les artefacts de l'expérience `GB_RET20_REFERENCE`. Il n'entraîne aucun modèle, n'accède pas au jeu de test et ne calcule aucune métrique sur le lockbox.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def trouver_racine_depot(depart: Path) -> Path:
    for candidat in (depart, *depart.parents):
        if (candidat / 'pyproject.toml').is_file():
            return candidat
    raise FileNotFoundError('Impossible de localiser la racine du dépôt.')


racine_depot = trouver_racine_depot(Path.cwd().resolve())
dossier_artefacts = racine_depot / 'artifacts' / 'experiments' / 'GB_RET20_REFERENCE'
oof = pd.read_csv(dossier_artefacts / 'oof_predictions.csv')
metriques_folds = pd.read_csv(dossier_artefacts / 'fold_metrics.csv')
metriques_groupes = pd.read_csv(dossier_artefacts / 'group_metrics.csv')
histogramme = pd.read_csv(dossier_artefacts / 'probability_histogram.csv')
with (dossier_artefacts / 'summary.json').open(encoding='utf-8') as fichier:
    resume = json.load(fichier)

## Identité et gate technique

In [ ]:
pd.Series({
    'expérience': resume['experiment_id'],
    'périmètre': resume['evaluation_scope'],
    'lignes OOF': len(oof),
    'hash OOF': resume['oof_sha256'],
    'gate technique': resume['gate']['technical_status'],
    'reproductible': resume['gate']['reproducible'],
})

## Métriques globales et par fold

In [ ]:
pd.Series(resume['global_metrics'])[['accuracy', 'roc_auc', 'log_loss', 'predicted_positive_rate', 'probability_mean', 'probability_std']]

In [ ]:
metriques_folds[['fold_id', 'n_validation', 'accuracy', 'roc_auc', 'log_loss', 'predicted_positive_rate']]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axe, colonne, titre in zip(
    axes,
    ['accuracy', 'roc_auc', 'log_loss'],
    ['Accuracy par fold', 'ROC-AUC par fold', 'Log-loss par fold'],
):
    metriques_folds.plot.bar(x='fold_id', y=colonne, ax=axe, legend=False, title=titre)
    axe.set_xlabel('Identifiant du fold')
plt.tight_layout()

## Matrice de confusion OOF

In [ ]:
confusion = resume['confusion_matrix']
pd.DataFrame(
    [[confusion['true_negative'], confusion['false_positive']],
     [confusion['false_negative'], confusion['true_positive']]],
    index=['Classe réelle 0', 'Classe réelle 1'],
    columns=['Classe prédite 0', 'Classe prédite 1'],
)

## Distribution des probabilités OOF

In [ ]:
largeurs = histogramme['bin_right'] - histogramme['bin_left']
plt.figure(figsize=(10, 4))
plt.bar(histogramme['bin_left'], histogramme['count'], width=largeurs, align='edge')
plt.axvline(0.5, color='black', linestyle='--', label='Seuil 0,5')
plt.xlabel('Probabilité prédite de la classe 1')
plt.ylabel('Nombre de lignes')
plt.title('Distribution des probabilités OOF sur le développement')
plt.legend()
plt.tight_layout()

## Dispersion descriptive par groupe TS

Ces statistiques décrivent uniquement les groupes de développement déjà présents dans les prédictions OOF.

In [ ]:
metriques_groupes[['accuracy', 'log_loss', 'predicted_positive_rate']].describe()

## Conclusion

La gate technique garantit la couverture OOF, l'intégrité des groupes, la validité des probabilités et la reproductibilité exacte des deux exécutions. Les avertissements scientifiques éventuels restent descriptifs et ne constituent pas une mesure du lockbox.